In [ ]:
import sys
from pathlib import Path

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
for _p in (str(_root), str(_root / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import scanpy as sc
import matplotlib.pyplot as plt

from metab_processing.metab_travlr_config import DATA_DIR
from metab_processing.SpaceTravLR.dataset_configs import dataset_paths
from metab_processing.LinearRegression.build_x import get_gene_factors
from metab_processing.LinearRegression.least_squares import (
    fit_gene_betas, rank_coefficients, plot_top_coefficients,
    subsample_gene_betas, plot_beta_histogram)

In [ ]:
data_dir = f'{DATA_DIR}/Alexi_UC_Spliced'
samples = [f'13473_HS4_UC-Slice_{i}' for i in (1, 2, 3, 4)]

ANNOT_COL = '25_06_11_ICI_5K_Coarse_annotations'
T_CELL   = 'IEC'
GLUCOSE  = 'D-Glucose'   # exact metabolite name (see inspect cell; may be folded into a merged name)
HIST_GENE = 'FOXP3'
SOURCE   = 'imputed'     # 'imputed' (MAGIC) or 'lognorm' (un-imputed log1p(raw); rebuild x_adata first)
METHOD, PENALTY, STANDARDIZE = 'l1', 0.001, True   # L1 + standardized -> tames the huge OLS betas
N_SUB, FRAC = 200, 0.8

def load_x(dataset):
    return sc.read_h5ad(dataset_paths(dataset, data_dir=data_dir)['dataset_dir'] / 'LinearRegression' / 'x_adata.h5ad')

In [ ]:
# Inspect one sample to set T_CELL / GLUCOSE / HIST_GENE above.
ad0 = load_x(samples[0])
print('annotations:', list(ad0.obs[ANNOT_COL].unique()))
print('metabolites:', [m.split('@', 1)[1] for m in ad0.uns['x_metab_modulators']])
print('genes:', list(ad0.uns['x_genes']))
print(f'{HIST_GENE} factors:', list(get_gene_factors(ad0, HIST_GENE, metabs='all', source=SOURCE).columns)[:20])

In [ ]:
# Visibility into the raw x (design-matrix) values behind any factor -- e.g. a TF and an L-R
# factor -- to see their scale (why unregularized betas blow up).
def factor_x(adata, gene, factor, source='imputed'):
    """The x values of one factor column for `gene` (from get_gene_factors)."""
    return get_gene_factors(adata, gene, metabs='all', source=source)[factor]

cols = list(get_gene_factors(ad0, HIST_GENE, metabs='all', source=SOURCE).columns)
tf_col = next((c for c in cols if not any(s in c for s in ('@', '$', '#'))), None)
lr_col = next((c for c in cols if '$' in c), None)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for col, ax in zip((tf_col, lr_col), axes):
    if col is None:
        continue
    ax.hist(factor_x(ad0, HIST_GENE, col, source=SOURCE), bins=40)
    ax.set_title(f'x values: {col}'); ax.set_xlabel('x')
fig.tight_layout()

In [ ]:
# T cells (one sample): L1 + standardized coefficients ranked by |magnitude|.
betas = fit_gene_betas(ad0, annot_col=ANNOT_COL, annot_value=T_CELL, source=SOURCE,
                       method=METHOD, penalty=PENALTY, standardize=STANDARDIZE)
display(rank_coefficients(betas).head(30))
plot_top_coefficients(betas, top=20).set_title(f'{samples[0]} - {T_CELL} ({METHOD}, standardized)');

In [ ]:
# All samples: top T-cell coefficients (L1 standardized, SOURCE block).
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for dataset, ax in zip(samples, axes.ravel()):
    b = fit_gene_betas(load_x(dataset), annot_col=ANNOT_COL, annot_value=T_CELL, source=SOURCE,
                       method=METHOD, penalty=PENALTY, standardize=STANDARDIZE)
    plot_top_coefficients(b, top=15, ax=ax); ax.set_title(f'{dataset} - {T_CELL}')
fig.tight_layout()

In [ ]:
# Cell subsampling: fit the FULL model each draw and return EVERY factor's beta
# (tf / lig$rec / lig#tf / metab@name), per sample. Compute once here; select below.
subsamp = {}
for dataset in samples:
    adx = load_x(dataset)
    gene = HIST_GENE or adx.uns['x_genes'][0]
    subsamp[dataset] = subsample_gene_betas(
        adx, gene, factors=None, n_subsamples=N_SUB, frac=FRAC,
        annot_col=ANNOT_COL, annot_value=T_CELL, source=SOURCE,
        method=METHOD, penalty=PENALTY, standardize=STANDARDIZE)
print({d: df.shape for d, df in subsamp.items()})

In [ ]:
# Select ANY factor (tf / lig$rec / lig#tf / metab@name) and see its subsampling distribution.
FACTOR = f'metab@{GLUCOSE}'
cols = subsamp[samples[0]].columns
if FACTOR not in cols:
    FACTOR = next((c for c in cols if c.startswith('metab@')), cols[0])
    print(f'metab@{GLUCOSE} not found; using {FACTOR!r}. Metab options:',
          [c for c in cols if c.startswith('metab@')][:10])
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for dataset, ax in zip(samples, axes.ravel()):
    plot_beta_histogram(subsamp[dataset][FACTOR], ax=ax, title=f'{dataset}\n{HIST_GENE} ~ {FACTOR}')
    ax.set_xlabel('beta')
fig.tight_layout()